In [ ]:
import os
import yaml
import json
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

from anngeno import AnnGeno
import multiprocessing

import matplotlib.pyplot as plt
from plotnine import *

num_cores = multiprocessing.cpu_count()
print(num_cores)

## Code

In [ ]:
def process_phenotypes_prs_long(
    gene_trait_df: pl.DataFrame,
    pheno_path: str,
    prs_path: str,
    cov_path: str,
    config: dict,
) -> pl.DataFrame:
    """
    Process phenotypes and PRS, compute residuals for each phenotype, and return a long-format Polars DataFrame.
    """
    print("Process phenotypes and PRS, compute residuals for each phenotype, and return a long-format Polars DataFrame.")
    
    # --- Step 1: Unique phenotypes ---
    unique_phenotypes = gene_trait_df['phenotype'].unique().to_list()
    if not unique_phenotypes:
        raise ValueError("No phenotypes found in gene_trait_df")

    # --- Step 2: Read data ---
    phenos = (
        pl.read_parquet(pheno_path)
        .rename({'IID': 'individual'})
        .select(['individual'] + unique_phenotypes)
        .drop_nulls()
    )

    prs_cols = [f"{pheno}_prs" for pheno in unique_phenotypes]
    prs = (
        pl.read_parquet(prs_path)
        .rename({'IID': 'individual'})
        .select(['individual'] + prs_cols)
        .drop_nulls()
    )

    cov_list = config.get("covariates", [])
    cov_df = (
        pl.read_parquet(cov_path)
        .rename({'sample': 'individual'})
        .select(['individual'] + cov_list)
        .with_columns(pl.col('individual').cast(pl.Int64))
    )

    # --- Step 3: Merge all into one DataFrame ---
    all_df = phenos.join(prs, on='individual', how='inner').join(
        cov_df, on='individual', how='inner'
    )
    all_pd = all_df.to_pandas()

    # --- Step 4: Compute residuals ---
    all_residuals_dfs = []

    for phenotype in unique_phenotypes:
        try:
            pheno_cols = [phenotype, f"{phenotype}_prs"] + cov_list
            temp_df = all_pd[['individual'] + pheno_cols].dropna()
            if len(temp_df) == 0:
                print(f"No data for phenotype: {phenotype}")
                continue

            y = temp_df[phenotype]
            X = temp_df.drop(columns=[phenotype, 'individual'])
            X = sm.add_constant(X)

            model = sm.OLS(y, X).fit()
            residuals = pd.Series(model.resid, index=temp_df.index, name=f"{phenotype}_residual")

            pheno_residuals = pd.concat([temp_df[['individual']], residuals], axis=1)
            all_residuals_dfs.append(pheno_residuals)

        except Exception as e:
            print(f"Error processing phenotype {phenotype}: {e}")
            continue

    if not all_residuals_dfs:
        raise ValueError("No residuals could be computed")

    # --- Step 5: Convert to long-format Polars DataFrame ---
    long_dfs = []
    for residual_df in all_residuals_dfs:
        p_wide = pl.DataFrame(residual_df).with_columns(
            pl.col('individual').cast(pl.String)
        )
        pheno_cols = [c for c in p_wide.columns if c.endswith('_residual')]
        pdf = (
            p_wide.unpivot(
                index=['individual'],
                on=pheno_cols,
                variable_name='phenotype',
                value_name='pheno_value',
            )
            .with_columns(
                pl.col('phenotype').str.replace('_residual', '').alias('phenotype')
            )
        )
        long_dfs.append(pdf)

    combined_pdf = pl.concat(long_dfs) if len(long_dfs) > 1 else long_dfs[0]

    return combined_pdf

def process_gene_genotypes(
    gene_id: str, 
    regions_dict: dict, 
    sample_list: list
) -> pl.DataFrame:
    """
    Extract genotypes for a specific gene
    """
    try:
        reg_dict = regions_dict[gene_id]
        geno = reg_dict['genotypes']
        
        # Find heterozygous genotypes (genotype == 1)
        rows, cols = np.where(geno == 1)
        het = pl.DataFrame({
            'id': np.array(reg_dict['variant_ids'])[rows],
            'individual': np.array(sample_list)[cols],
            'genotype': 1
        })
        
        # Find homozygous genotypes (genotype == 2)
        rows, cols = np.where(geno == 2)
        hom = pl.DataFrame({
            'id': np.array(reg_dict['variant_ids'])[rows],
            'individual': np.array(sample_list)[cols],
            'genotype': 2
        })
        
        geno_melt = pl.concat([het, hom]).with_columns(
            pl.lit(gene_id).alias('region')
        )
        
        return geno_melt
        
    except Exception as e:
        print(f"Error processing genotypes for gene {gene_id}: {e}")
        return None, None

def bootstrap_samples(
    gpa_df: pl.DataFrame, 
    n_bootstraps: int, 
    seed: int = None
) -> pl.DataFrame:
    """
    Bootstrap samples for correlation analysis
    """
    if seed is not None:
        np.random.seed(seed)

    # Get unique individuals
    unique_ids = gpa_df.select('individual').unique().to_series().to_list()
    n_ids = len(unique_ids)

    # Prepare all bootstrap samples
    sampled_ids = np.random.choice(unique_ids, size=(n_bootstraps, n_ids), replace=True)

    # Flatten and make DataFrame with bootstrap_id
    boot_id_col = np.repeat(np.arange(n_bootstraps), n_ids)
    sampled_flat = pl.LazyFrame({
        'bootstrap_id': boot_id_col,
        'individual': sampled_ids.ravel()
    })

    # Lazy join to replicate rows
    boot_df = sampled_flat.join(gpa_df.lazy(), on='individual', how='left')

    # Group by bootstrap_id + original grouping columns
    result = (
        boot_df
        .group_by(['bootstrap_id', 'id', 'phenotype', 'region', 'annotation'])
        .agg([
            pl.len().alias('n_individuals'),
            pl.col('pheno_value').mean().alias('mean_pheno_value'),
            pl.col('score').mean().alias('mean_score'),
        ])
        .collect()
    )

    return result

In [ ]:
def corr_pipeline(
    gene_trait_df: pl.DataFrame,
    anngeno_path: str,
    pheno_path: str,
    prs_path: str,
    cov_path: str,
    new_annotations_path: str,
    config: dict,
    maf: float = None,
    eur_samples_path: str = None,
    corr_method: str = "pearson",
    save_path: str = None,
    only_missense_vars: bool = False,
    all_vars: bool = False,
):
    """
    Main processing pipeline for gene-trait correlation analysis.
    """

    # Process phenotypes (PRS + covariate correction)
    combined_pdf = process_phenotypes_prs_long(
        gene_trait_df,
        pheno_path,
        prs_path,
        cov_path,
        config,
    )

    # Initialize AnnGeno
    print("Loading AnnGeno...")
    ag = AnnGeno(anngeno_path, mode='r', low_mem=True)

    if maf:
        variants_to_keep = (
            ag.annotations.filter(pl.col('af_ukb') <= maf)
            .select('id')
            .collect()['id']
        )
        ag.subset_variants(set(variants_to_keep))

    eur_samples = None
    if eur_samples_path:
        eur_samples = pl.read_csv(eur_samples_path).with_columns(
            pl.col("eid").cast(pl.Utf8)
        )['eid'].to_list()
        ag.subset_samples(set(eur_samples))

    # Get regions
    unique_genes = gene_trait_df['gene_id'].unique().to_list()
    regions_dict = ag.get_many_regions(unique_genes)

    # Load new scores
    if new_annotations_path:
        print("Adding new scores to benchmark...")
        new_anno = pl.read_parquet(new_annotations_path).filter(pl.col('region').is_in(unique_genes))

    # Collect annotation categories
    all_annotation_list = []
    rare_variant_annotations_dict = config.get("rare_variant_annotations")
    if rare_variant_annotations_dict:
        for category in rare_variant_annotations_dict.values():
            all_annotation_list.extend(category)

    all_results = []
    pheno_gis_df = None  # last computed pheno_gis_df, useful when plotting for a single gene-trait pair

    gene_to_traits = (
        gene_trait_df.group_by("gene_id")
        .agg(pl.col("phenotype").unique().alias("phenotypes"))
    )

    # --- Loop by gene, compute genotypes once ---
    for row in tqdm(gene_to_traits.iter_rows(named=True), total=len(gene_to_traits)):
        gene_id = row["gene_id"]
        phenotypes = row["phenotypes"]

        print(f"Processing gene {gene_id} with {len(phenotypes)} phenotypes")

        geno_melt = process_gene_genotypes(gene_id, regions_dict, ag.samples)
        if geno_melt is None:
            print(f"No genotype data for {gene_id}")
            continue

        anno_df = regions_dict[gene_id]["annotations"]

        # TODO!
        if new_annotations_path:
            new_anno_gene = new_anno.filter(pl.col("region") == gene_id).pivot(
                index=['mutant', 'gene_id', 'gene_name'],
                on='dms_type',
                values='dms_score'
            ).drop_nulls()

        # --- Loop over phenotypes for this gene ---
        for phenotype in phenotypes:
            print(f" -> {gene_id} - {phenotype}")

            pheno_data = combined_pdf.filter(pl.col("phenotype") == phenotype)
            if len(pheno_data) == 0:
                print(f"No phenotype data for {phenotype}")
                continue

            gp_df = geno_melt.join(pheno_data, on="individual")

            if eur_samples is not None:
                gp_df = gp_df.filter(pl.col("individual").is_in(eur_samples))

            gp_df = gp_df.filter(pl.col("genotype") == 1)
            if len(gp_df) == 0:
                print(f"No valid genotype-phenotype data for {gene_id} - {phenotype}")
                continue

            if new_annotations_path:
                if 'id' in new_anno_gene.columns:
                    anno_wide = anno_df.join(new_anno_gene, on='id', how='inner')
                elif 'mutant' in new_anno_gene.columns:
                    anno_wide = anno_df.join(new_anno_gene, on='mutant', how='inner')
                else:
                    print(f"No matching column for joining new annotations for {gene_id}")
                    continue
                available_annotations = list(set(all_annotation_list) & set(anno_wide.columns)) + new_anno_gene.columns[3:]
            else:
                anno_wide = anno_df
                available_annotations = list(set(all_annotation_list) & set(anno_wide.columns))

            if not available_annotations:
                print(f"No valid annotations for {gene_id}")
                continue

            if only_missense_vars:
                anno_wide = anno_wide.filter(pl.col('consequence_missense_variant') == 1)

            anno_melt = anno_wide.unpivot(
                index=['id', 'region', 'af_ukb'],
                on=available_annotations,
                variable_name='annotation',
                value_name='score',
            ).with_columns(
                pl.col('score').cast(pl.Float32).alias('score')
            )

            ## Add all variant scores
            if all_vars:
                annos_all_vars = list(set(all_annotation_list) & set(anno_df.columns))
                all_anno_melt = anno_df.filter(pl.col('consequence_missense_variant') == 1).unpivot(
                    index=['id', 'region'],
                    on=annos_all_vars,
                    variable_name='annotation',
                    value_name='score',
                ).with_columns(
                    (pl.col('annotation') + "_allvars").alias('annotation')
                )
                anno_melt = pl.concat([anno_melt, all_anno_melt])

            gpa_df = gp_df.join(anno_melt, on='id', how='inner')
            if len(gpa_df) == 0:
                print(f"No data after joining annotations for {gene_id} - {phenotype}")
                continue

            # --- correlation step ---
            pheno_gis_df = (
                gpa_df.group_by(['id', 'phenotype', 'region', 'annotation'])
                .agg([
                    pl.len().alias('n_individuals'),
                    pl.col('pheno_value').mean().alias('mean_pheno_value'),
                    pl.col('score').mean().alias('score'),
                ])
                .drop_nulls(subset=['mean_pheno_value', 'score'])
            )
            if len(pheno_gis_df) == 0:
                continue

            corr_df = (
                pheno_gis_df.group_by(['annotation', 'phenotype', 'region'])
                .agg(pl.corr('score', 'mean_pheno_value', method=corr_method).alias('corr'))
                .with_columns(pl.col('corr').abs().alias('abs_corr'))
                .filter(pl.col('abs_corr').is_not_null())
                .sort('abs_corr', descending=True)
                .with_columns([
                    pl.lit(gene_id).alias('gene_id'),
                    pl.lit(phenotype).alias('phenotype_name'),
                ])
            )

            all_results.append(corr_df)

    # Combine results
    final_corr_df = pl.concat(all_results)
    print(f"Final correlation results shape: {final_corr_df.shape}")
    if save_path:
        final_corr_df.write_parquet(f'{save_path}/multi_gene_trait_correlations.parquet')

    return pheno_gis_df, final_corr_df


### Define params

In [ ]:
# Configuration and paths
corr_method = 'spearman'
bootstrapping = False
trait_type = 'quantitative'
mac = 10

config_path = f'/home/dnanexus/ukbgym/config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

pheno_path = '/home/dnanexus/data_dir/phenotypes/phenotypes190_missing20_unique2_int.parquet'
prs_path = '/home/dnanexus/data_dir/phenotypes/PRS190_EUR_missing20_unique2_int.parquet'
cov_path = '/home/dnanexus/data_dir/phenotypes/250709_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet'
eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'

anngeno_path = '/home/dnanexus/data_dir/genebass_1e6_coding_variants.ag'
# anngeno_path = '/home/dnanexus/data_dir/dms_coding.ag'
# exp_data_path = '/home/dnanexus/data_dir/exp_data/250819_Beltran_ssm_scores.parquet'
# exp_data_path = '/home/dnanexus/data_dir/exp_data/250717_proteinGym_SNP_96DMS.parquet'
save_path = None

In [ ]:
n = !wc -l $eur_samples_path
n_samples = int(n[0].split(' ')[0])
n_samples

In [ ]:
maf = mac/(2*n_samples)
maf

### DMS data

In [ ]:
pg = pl.read_parquet('/home/dnanexus/data_dir/exp_data/250717_proteinGym_SNP_96DMS.parquet')
pg

In [ ]:
a = pg.filter(pl.col('gene_name')=='NPC1').pivot(
    index=['mutant', 'gene_id', 'gene_name'],
    on='dms_type',
    values='dms_score'
)
a.drop_nulls()

In [ ]:
many_pg = pg[['file_name', 'gene_name']].unique()['gene_name'].value_counts(sort=True).filter(pl.col('count') > 1)
many_pg

In [ ]:
t = pg[['file_name', 'gene_name']].unique().join(pg['file_name'].value_counts(sort=True), on='file_name')
t

In [ ]:
lm = pl.read_parquet('/home/dnanexus/data_dir/exp_data/250822_Livesey_Marsh_36DMS.parquet')
a = lm[['fitness_assay', 'gene_name', 'reference']].unique().join(lm['gene_name'].value_counts(sort=True), on='gene_name')
a

## Exectute pipeline

In [ ]:
# an = pl.read_parquet(f'{anngeno_path}/annotations.parquet')
# an.columns = [col.lower() for col in an.columns]
# # Remove "cadd" prefix from all columns except "cadd_raw" and "cadd_phred"
# cols = []
# for col in an.columns:
#     if col.startswith("cadd") and col not in ["cadd_raw", "cadd_phred"]:
#         new_col = col.replace("cadd_", "")
#         cols.append(new_col)
#     else:
#         cols.append(col)
# an.columns = cols

# # Drop columns with the suffix "right"
# an = an.drop([col for col in an.columns if col.endswith("right")])
# an.write_parquet(f'{anngeno_path}/annotations.parquet')
an = pl.scan_parquet(f'{anngeno_path}/annotations.parquet')
an_cols = an.collect_schema().names()
an_genes = an.select('region').unique().collect()['region'].to_list()

ph_cols = pl.scan_parquet(pheno_path).collect_schema().names()
ph_cols

In [ ]:
a = pl.read_parquet('/home/dnanexus/data_dir/genebass_continuous_associations_ukbbgym.pq').with_columns(
    (pl.col('description').str.replace_all(r"\(", "").str.replace_all(r"\)", "").str.replace_all(' ', '_').str.to_lowercase() + '_int').alias('phenotype'),
    pl.col('gene_id').alias('region')
).filter(
    pl.col('annotation').str.contains('pLoF'),
    pl.col('phenotype').is_in(ph_cols),
    pl.col('region').is_in(an_genes)
)

# gene_trait_df = a[['region', 'gene_id', 'gene_symbol', 'phenotype', 'trait_type']].unique().filter(pl.col('gene_symbol')=='GCK')
gene_trait_df = a[['region', 'gene_id', 'gene_symbol', 'phenotype', 'trait_type']].unique()
gene_trait_df

In [ ]:
def split_genes_and_df(gene_trait_df, n_splits):
    genes = gene_trait_df['gene_id'].unique().sort()
    genes_per_split = int(len(genes) / n_splits)
    genes_splits = [genes[i*genes_per_split:(i+1)*genes_per_split] for i in range(n_splits-1)]
    genes_splits.append(genes[(n_splits-1)*genes_per_split:])  # last split gets the remainder

    df_splits = [gene_trait_df.filter(pl.col('gene_id').is_in(g)) for g in genes_splits]
    return genes_splits, df_splits

# Example usage:
genes_splits, assoc_df_splits = split_genes_and_df(gene_trait_df, 5)
for df in assoc_df_splits:
    print(df.shape)

In [ ]:

_, corr_df1 = corr_pipeline(
    gene_trait_df=assoc_df_splits[0],
    anngeno_path=anngeno_path,
    pheno_path=pheno_path,
    prs_path=prs_path,
    cov_path=cov_path,
    new_annotations_path=None,
    config=config,
    maf=maf,
    eur_samples_path=eur_samples_path,
    corr_method=corr_method,
    save_path=save_path,
    only_missense_vars=False,
    all_vars=False,
)

corr_df1

In [ ]:
# Invert the mapping: annotation -> category
annotation_to_cat = {
    ann: cat
    for cat, anns in config['rare_variant_annotations'].items()
    for ann in anns
}

# Add mapped category column (unmatched will be null)
corr_df = corr_df1.with_columns(
    pl.col("annotation")
      .replace(annotation_to_cat, default=None)  # unmatched → None
      .alias("category")
)

corr_df = corr_df.with_columns(
    pl.col('annotation').str.split('_').list.get(-1).alias('aggregation')
)

med_df = corr_df.drop_nans().group_by("annotation").agg([
    pl.col("abs_corr").median().alias("median_abs_corr")
])

corr_pd = med_df.join(corr_df, on='annotation').to_pandas()

corr_pd['annotation'] = pd.Categorical(
    corr_pd['annotation'],
    categories=corr_pd.sort_values('median_abs_corr', ascending=False)['annotation'].unique(),
    ordered=True
)

# Plot
(
    ggplot(corr_pd, aes(x='annotation', y='abs_corr', fill='annotation')) +
    geom_boxplot(alpha=0.8) +
    theme_minimal() +
    facet_wrap('category', scales='free') +
    # scale_y_sqrt() +
    coord_flip() +
    labs(
        x='Annotation',
        y='Absolute Spearman correlation'
    ) +
    theme(
        figure_size=(14, 6),
        legend_position='none'
    )
)

## Single gene-trait plot

In [ ]:
corr_df.drop_nans().filter(pl.col('annotation')=='am_pathogenicity').sort('abs_corr', descending=True)#[0]['phenotype'].item()

In [ ]:
gene_trait_df = pl.DataFrame({
    'gene_id': ["ENSG00000163554"],#["ENSG00000141867"],
    'phenotype': ["mean_reticulocyte_volume_int"] #['jurgens_bipolar_disorder']
})

plot_df_gene, corr_results_gene, _ = corr_pipeline(
    gene_trait_df=gene_trait_df,
    anngeno_path=anngeno_path,
    pheno_path=pheno_path,
    prs_path=prs_path,
    cov_path=cov_path,
    exp_data_path=exp_data_path,
    config=config,
    maf=maf,
    eur_samples_path=eur_samples_path,
    corr_method=corr_method,
    bootstrapping=bootstrapping,
    save_path=save_path
)

plot_df_gene

In [ ]:
import sys
from IPython.display import display

def plot_correlation(plot_df, phenotype, gene_id, annotation, method='spearman'):
    # Filter and add ranks
    df_filtered = plot_df.filter(
        (pl.col('phenotype') == phenotype) &
        (pl.col('region') == gene_id) &
        (pl.col('annotation') == annotation)
    ).with_columns([
        pl.col('score').rank().alias('score_rank'),
        pl.col('mean_pheno_value').rank().alias('pheno_rank')
    ])

    # Determine columns to correlate and plot
    if method.lower() == 'spearman':
        x_col, y_col = 'score_rank', 'pheno_rank'
    elif method.lower() == 'pearson':
        x_col, y_col = 'mean_score', 'mean_pheno_value'
    else:  # Pearson
        sys.exit(f"Unrecognized method. Use 'spearman' or 'pearson'.")

    # Compute correlation using Polars
    corr = df_filtered.select([pl.corr(x_col, y_col, method='pearson')]).to_numpy()[0, 0]

    corr_text = f"{method.title()} r = {corr:.2f}"

    # Build plot
    rc_plot = (
        ggplot(df_filtered.to_pandas(), aes(x=x_col, y=y_col)) +
        geom_point(alpha=0.25) +
        geom_smooth(method='lm', se=True, color='darkred') +
        theme_minimal() +
        labs(
            x=f"{annotation} {'rank' if method.lower()=='spearman' else ''}",
            y=f"{phenotype} residual {'rank' if method.lower()=='spearman' else ''}"
        ) +
        annotate(
            'text',
            x=df_filtered[x_col].min(),
            y=df_filtered[y_col].max(),
            label=corr_text,
            ha='left',
            va='top',
            size=12
        ) +
        theme(figure_size=(5, 4))
    )

    return rc_plot

In [ ]:
corr_method = 'spearman'
am_plot = plot_correlation(plot_df_gene, gene_trait_df['phenotype'].item(), gene_trait_df['gene_id'].item(), 'am_pathogenicity', method=corr_method)

display(am_plot)

In [ ]:
plot_df_gene['annotation'].value_counts(sort=True)